# Modelling (Machine Learning & Deep Learning)

Terdapat 4 skenario pemodelan utama:
1. **SVM Tanpa Augmentasi** (+ Ekstraksi Fitur)
2. **SVM Dengan Augmentasi** (+ Ekstraksi Fitur)
3. **MobileNetV2 (Pre-trained) Tanpa Augmentasi**
4. **MobileNetV2 (Pre-trained) Dengan Augmentasi**

> **Adaptive Logic:** Notebook ini secara pintar akan mendeteksi apakah data hasil augmentasi dari *Notebook 02* sudah tersedia di disk lokal (`data/results/augmented/`). Jika ada, model akan sekadar meload data tersebut (jauh lebih cepat). Jika tidak ada (misalnya jika di-*run* ulang di Kaggle Notebook dari nol), model akan melakukan praproses dan augmentasi secara *on-the-fly*.

In [ ]:
import os
import joblib
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
import kagglehub

# Scikit-Learn (ML Klasik)
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

# TensorFlow / Keras (Deep Learning)
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Input
from tensorflow.keras.preprocessing.image import ImageDataGenerator



## Setup Direktori

In [ ]:
path = kagglehub.dataset_download("ashishjangra27/face-mask-12k-images-dataset")
KAGGLE_DIR = Path(path) / "Face Mask Dataset"
if not KAGGLE_DIR.exists(): KAGGLE_DIR = Path(path)

RESULTS_ROOT = Path("../data/results")
FEATURES_DIR = Path("../data/features")
CLASSES = ["WithMask", "WithoutMask", "MaskWornIncorrect"]

AUGMENTED_DIR = RESULTS_ROOT / "augmented" / "Train"
has_saved_data = AUGMENTED_DIR.exists()
print("Apakah data lokal hasil augmentasi tersedia? :", has_saved_data)

## Skenario 1 & 2: SVM (ML Klasik)

In [ ]:
print("Loading Fitur Canny untuk Skenario 1 & 2 dari disk...")
try:
    # Load Unaugmented (Skenario 1)
    data_unaug = np.load('../data/features/canny_unaug_features.npz')
    X_train_unaug, y_train_unaug = data_unaug['X_train'], data_unaug['y_train']
    X_val, y_val = data_unaug['X_val'], data_unaug['y_val']
    
    # Load Augmented (Skenario 2)
    data_aug = np.load('../data/features/canny_aug_features.npz')
    X_train_aug, y_train_aug = data_aug['X_train'], data_aug['y_train']
    
    # Training Skenario 1 (Tanpa Aug)
    svm_unaug = SVC(kernel='rbf')
    svm_unaug.fit(X_train_unaug, y_train_unaug) # Sudah di-scale di notebook 03
    pred_unaug = svm_unaug.predict(X_val)
    print("\n--- Skenario 1: SVM Tanpa Augmentasi ---")
    print("Accuracy:", accuracy_score(y_val, pred_unaug))
    os.makedirs('../models', exist_ok=True)
    joblib.dump(svm_unaug, '../models/svm_unaug.pkl')
    
    # Training Skenario 2 (Dengan Aug)
    svm_aug = SVC(kernel='rbf')
    svm_aug.fit(X_train_aug, y_train_aug) # Sudah di-scale di notebook 03
    pred_aug = svm_aug.predict(X_val)
    print("\n--- Skenario 2: SVM Dengan Augmentasi ---")
    print("Accuracy:", accuracy_score(y_val, pred_aug))
    joblib.dump(svm_aug, '../models/svm_aug.pkl')
    
except FileNotFoundError:
    print("File .npz tidak ditemukan! Pastikan Anda menjalankan Notebook 03 terlebih dahulu untuk men-generate fitur ML Klasik.")


## Skenario 3 & 4: Deep Learning (MobileNetV2)
Memiliki adaptabilitas: jika ada file lokal, panggil yang lokal. Jika tidak ada, augmentasi on-the-fly.

In [ ]:
def mobilenet_preprocessing(img_array):
    """Untuk memproses gambar mentah langsung dari Kaggle"""
    img = img_array.astype(np.uint8)
    if img.shape[-1] == 3: img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    img = clahe.apply(img)
    img = cv2.GaussianBlur(img, (5, 5), 0)
    img_3_channel = np.stack((img,)*3, axis=-1)
    return preprocess_input(img_3_channel.astype(np.float32))

def mobilenet_preprocessing_saved(img_array):
    """Untuk memproses gambar yang SUDAH jadi grayscale di disk (di-load sbg RGB oleh Keras)"""
    img = img_array.astype(np.uint8)
    img_1_channel = img[:,:,0]
    img_3_channel = np.stack((img_1_channel,)*3, axis=-1)
    return preprocess_input(img_3_channel.astype(np.float32))


In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Skenario 3 (Tanpa Augmentasi, dari raw Kaggle)
datagen_unaug = ImageDataGenerator(preprocessing_function=mobilenet_preprocessing)

# Skenario 4 (Dengan Augmentasi: Adaptif)
if has_saved_data:
    print("Skenario 4: Menggunakan data tersimpan (Praproses & Augmentasi sudah ada di disk).")
    datagen_aug = ImageDataGenerator(preprocessing_function=mobilenet_preprocessing_saved)
    train_gen_aug_path = AUGMENTED_DIR
else:
    print("Skenario 4: Menjalankan augmentasi secara on-the-fly (Cocok untuk cloud notebook).")
    datagen_aug = ImageDataGenerator(
        rotation_range=10,
        width_shift_range=0.2,
        height_shift_range=0.2,
        zoom_range=0.25,
        horizontal_flip=True,
        preprocessing_function=mobilenet_preprocessing
    )
    train_gen_aug_path = KAGGLE_DIR / "Train"

train_gen_unaug = datagen_unaug.flow_from_directory(
    KAGGLE_DIR / "Train", target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical')

train_gen_aug = datagen_aug.flow_from_directory(
    train_gen_aug_path, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical')

val_gen = datagen_unaug.flow_from_directory(
    str(KAGGLE_DIR / "Validation"), 
    target_size=IMG_SIZE, 
    batch_size=BATCH_SIZE, 
    class_mode='categorical',
    shuffle=False
)



In [ ]:
def build_mobilenet_model():
    base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    for layer in base_model.layers:
        layer.trainable = False
        
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.5)(x)
    predictions = Dense(2, activation='softmax')(x)
    
    model = Model(inputs=base_model.input, outputs=predictions)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

model_unaug = build_mobilenet_model()
history_unaug = model_unaug.fit(train_gen_unaug, validation_data=val_gen, epochs=5)
os.makedirs('../models', exist_ok=True)
model_unaug.save('../models/mobilenet_unaug.h5')

model_aug = build_mobilenet_model()
history_aug = model_aug.fit(train_gen_aug, validation_data=val_gen, epochs=5)
model_aug.save('../models/mobilenet_aug.h5')